# Agentic Email Marketing Service
Built with CrewAI, SendGrid, and Gradio.

### Import relevant libraries

In [ ]:
import os
import json
import io
import asyncio
import gradio as gr
import sendgrid
from typing import List, Optional
from pydantic import BaseModel, Field
from duckduckgo_search import DDGS
from sendgrid.helpers.mail import Content, Email, Mail, To
from crewai import Agent, Task, Crew
from crewai.tools import tool
from dotenv import load_dotenv
from rich.console import Console

load_dotenv(override=True)

console = Console(force_terminal=True, width=80)
_trace_console: Optional[Console] = None


def _active_console() -> Console:
    return _trace_console if _trace_console is not None else console

### Set up environment

In [ ]:
SENDER_EMAIL = "answers@ailysis.io"

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
SENDGRID_API_KEY = os.environ.get("SENDGRID_API_KEY")

print("Environment ready.")

### Business Tools

In [ ]:
def _web_search_impl(query: str) -> str:
    """Search the web for market trends and competitor info."""
    _active_console().print(f"[bold cyan]Researcher is searching for:[/bold cyan] {query}")
    try:
        with DDGS() as ddgs:
            results = [r for r in ddgs.text(query, max_results=3)]
        return json.dumps(results) if results else "No specific results found for this query."
    except Exception as e:
        return f"Search error: {str(e)}"

def _fetch_subscriber_emails_impl() -> List[str]:
    """Registered subscriber emails."""
    emails = ["xxxx@gmail.com", "xxxxx@gmail.com"]
    _active_console().print(
        f"[bold magenta]Database:[/bold magenta] Retrieved {len(emails)} subscribers."
    )
    return emails

def _send_marketing_email_impl(subject: str, html_content: str, recipients: List[str]) -> str:
    """Send via SendGrid."""
    _active_console().print(
        f"[bold green]Dispatching email to {len(recipients)} recipient(s)...[/bold green]"
    )
    api_key = os.environ.get("SENDGRID_API_KEY")
    if not api_key:
        return "Skipped send: SENDGRID_API_KEY is not set (dry run)."
    
    try:
        sg = sendgrid.SendGridAPIClient(api_key=api_key)
        from_email = Email(SENDER_EMAIL)
        for email_addr in recipients:
            to_email = To(email_addr)
            content = Content("text/html", html_content)
            mail = Mail(from_email, to_email, subject, content)
            response = sg.client.mail.send.post(request_body=mail.get())
            if response.status_code not in (200, 201, 202):
                return f"SendGrid error {response.status_code}: {response.body}"
        return f"Successfully sent to {len(recipients)} recipients."
    except Exception as e:
        return f"Error: {str(e)}"

@tool("web_search")
def web_search(query: str) -> str:
    """Search the web for market trends and competitor info. Query must be a specific search string."""
    return _web_search_impl(query)

def fetch_subscriber_emails() -> List[str]:
    """Queries the database for registered user emails."""
    return _fetch_subscriber_emails_impl()

def send_marketing_email(subject: str, html_content: str, recipients: List[str]) -> str:
    """Sends the finalized marketing email via SendGrid."""
    return _send_marketing_email_impl(subject, html_content, recipients)


### Agents and orchestrated pipeline

In [ ]:
HOW_MANY_SEARCHES = 3

class WebSearchItem(BaseModel):
    reason: str = Field(description="Why this search matters for the query.")
    query: str = Field(description="The search term to use for the web search.")

class WebSearchPlan(BaseModel):
    searches: List[WebSearchItem] = Field(
        description="Web searches to perform to best answer the marketing research query."
    )

class EmailDraft(BaseModel):
    subject: str = Field(description="Catchy subject line.")
    html_body: str = Field(description="The marketing email in HTML format.")

planner_agent = Agent(
    role="Research Planner",
    goal="Propose web searches to gather trends, competitors, and positioning.",
    backstory="An expert marketing research assistant capable of finding the best strategies.",
    llm="gpt-4o-mini",
    allow_delegation=False
)

researcher = Agent(
    role="Market Researcher",
    goal="Find market trends and competitor positioning for the campaign.",
    backstory="A meticulous researcher who uses the web to find up-to-date market information.",
    tools=[web_search],
    llm="gpt-4o-mini",
    allow_delegation=False,
    max_iter=5
)

copywriter = Agent(
    role="Copywriter",
    goal="Turn research into one high-converting HTML marketing email.",
    backstory="An expert marketing copywriter specializing in high-converting email campaigns.",
    llm="gpt-4o-mini",
    allow_delegation=False
)

async def plan_searches(query: str) -> WebSearchPlan:
    task = Task(
        description=f"Given a marketing campaign brief, propose {HOW_MANY_SEARCHES} distinct web searches. Query: {query}",
        expected_output=f"Exactly {HOW_MANY_SEARCHES} search items.",
        agent=planner_agent,
        output_pydantic=WebSearchPlan
    )
    crew = Crew(agents=[planner_agent], tasks=[task], verbose=False)
    result = await crew.kickoff_async()
    return result.pydantic

async def search_item(item: WebSearchItem) -> str:
    task = Task(
        description=f"Focus only on this search term: {item.query}. Reason: {item.reason}. Call the web_search tool, wait for the result, and summarize key findings.",
        expected_output="A synthesis of the findings for this specific search query.",
        agent=researcher
    )
    crew = Crew(agents=[researcher], tasks=[task], verbose=False)
    result = await crew.kickoff_async()
    return result.raw

async def perform_searches(search_plan: WebSearchPlan) -> list[str]:
    tasks = [asyncio.create_task(search_item(item)) for item in search_plan.searches]
    return list(await asyncio.gather(*tasks))

async def write_email_draft(query: str, search_results: list[str]) -> Optional[EmailDraft]:
    task = Task(
        description=f"Turn research into a high-converting HTML marketing email. Output match EmailDraft schema.\n\nQuery: {query}\nResults: {search_results}",
        expected_output="The final marketing email draft containing a subject and HTML body.",
        agent=copywriter,
        output_pydantic=EmailDraft
    )
    crew = Crew(agents=[copywriter], tasks=[task], verbose=False)
    result = await crew.kickoff_async()
    return result.pydantic


### Streaming logic

In [ ]:
async def run_automation_stream(user_prompt: str):
    """Gradio stream: (plan → search → write → send)."""
    global _trace_console
    buf = io.StringIO()
    capture_console = Console(file=buf, force_terminal=True, width=100)
    _trace_console = capture_console

    def emit() -> str:
        return f"```ansi\n{buf.getvalue()}\n```"

    try:
        capture_console.print("[bold bright_cyan]Starting research flow...[/bold bright_cyan]")
        yield emit()

        capture_console.print("[yellow]Planning searches...[/yellow]")
        yield emit()
        search_plan = await plan_searches(user_prompt)
        
        if not search_plan or not search_plan.searches:
             capture_console.print("[bold red]Planning phase yielded no results.[/bold red]")
             yield emit()
             return

        capture_console.print(f"[bold green]Executing {len(search_plan.searches)} search tasks...[/bold green]")
        yield emit()
        search_results = await perform_searches(search_plan)
        capture_console.print("[bold green]Research complete.[/bold green]")
        yield emit()

        capture_console.print("[magenta]Generating email draft...[/magenta]")
        yield emit()
        draft = await write_email_draft(user_prompt, search_results)
        
        if draft is None:
            capture_console.print("[bold red]Email drafting failed.[/bold red]")
            yield emit()
            return
        
        capture_console.print("[bold green]Draft created successfully.[/bold green]")
        yield emit()

        recipients = _fetch_subscriber_emails_impl()
        capture_console.print("[yellow]Sending to subscribers...[/yellow]")
        yield emit()
        send_status = _send_marketing_email_impl(
            draft.subject, draft.html_body, recipients
        )
        capture_console.print(f"[dim]{send_status}[/dim]")
        
        if send_status.startswith("Successfully"):
            capture_console.print("[bold bright_yellow]Success: Campaign Dispatched![/bold bright_yellow]")
        else:
            capture_console.print("[bold red]Dispatch failed.[/bold red]")
        yield emit()

    except Exception as e:
        capture_console.print(f"\n[bold red]Critical system error:[/bold red] {str(e)}")
        yield emit()
    finally:
        _trace_console = None


### UI with execution trace

In [ ]:
def launch_ui():
    with gr.Blocks(theme=gr.themes.Default(), title="AI Marketing Service") as demo:
        gr.Markdown("# Agentic Email Marketing Service")
        
        with gr.Row():
            with gr.Column(scale=1):
                prompt_input = gr.Textbox(
                    label="Campaign Objective", 
                    placeholder="e.g., Promote our new eco-friendly sneakers...",
                    lines=5
                )
                launch_btn = gr.Button("Execute Full Flow", variant="primary")
            
            with gr.Column(scale=2):
                trace_output = gr.Markdown(
                    label="Real-time Execution Trace",
                    sanitize_html=False,
                )

        launch_btn.click(
            fn=run_automation_stream,
            inputs=prompt_input,
            outputs=trace_output,
            show_progress="minimal",
        )

    demo.launch(share=True)


if __name__ == "__main__":
    launch_ui()